
# Laboratory 3B — Energy–volume equation of state

We now move away from the static equilibrium geometry on purpose.

At each selected volume, CRYSTAL evaluates the minimum energy compatible with
that volume. Quantas then converts the resulting CRYSTAL outputs into an
energy–volume dataset and fits a third-order Birch–Murnaghan equation of state.


In [ ]:

from pathlib import Path

from mineral_science_tools import (
    eos_teaching_table,
    fit_bm3_with_quantas,
    make_cubic_volume_series,
    plot_energy_volume,
    plot_pressure_volume,
    read_quantas_ev_dataset,
    write_file_list,
)

# Copy the optimized conventional cubic lattice parameter from Notebook 3A.
A0_ANGSTROM = 4.212

VOLUME_FACTORS = (0.91, 0.94, 0.97, 1.00, 1.03, 1.06, 1.09)



## 1. Calculate the target lattice parameters

For cubic MgO,

$$
a(f)=a_0 f^{1/3},
$$

where $f=V/V_0$.


In [ ]:

targets = make_cubic_volume_series(A0_ANGSTROM, VOLUME_FACTORS)
targets



### Question 1

Why do we use $f^{1/3}$ to scale the lattice parameter rather than multiplying
$a_0$ directly by the volume factor $f$?

**Your answer:**  
_Write 2–3 lines here._



## 2. Prepare the constant-volume inputs

The file:

```text
inputs/mgo_cvopt_template.d12
```

contains:

```text
OPTGEOM
CVOLOPT
...
END
```

Create one copy for every non-equilibrium factor:

```text
0.91
0.94
0.97
1.03
1.06
1.09
```

The $1.00$ point is the completed full optimization from Notebook 3A.

For each copy:

1. replace the lattice parameter with the value calculated in the table above;
2. use the same SHRINK value chosen in Laboratory 2;
3. change nothing else.

On CluMiner, small changes can be made directly with Vim.



### Question 2

Why do we use `CVOLOPT` rather than another `FULLOPTG` calculation for these
structures?

What would happen to our intended energy–volume series if the volume were
allowed to relax freely?

**Your answer:**  
_Write 3–4 lines here._



## 3. Run the six additional calculations

For every input use:

```bash
cluminer-crystal INPUT.d12 --cores-per-node 2 --threads 2
```

Retrieve the outputs into:

```text
work/lab03/outputs/
```

Keep the resource request fixed for the entire series.



### Question 3

Inspect one constant-volume output. How many structural degrees of freedom
remain for cubic MgO once the volume has been fixed?

Use the optimization history in the output as evidence.

**Your answer:**  
_Write 2–3 lines here._



## 4. Tell Quantas which outputs belong to the EOS series

Edit the filenames below so that they match your files exactly.

The central point is the final full geometry-optimization output.


In [ ]:

OUTPUTS = [
    Path("outputs/mgo_v091.out"),
    Path("outputs/mgo_v094.out"),
    Path("outputs/mgo_v097.out"),
    Path("outputs/mgo_fullopt.out"),   # V/V0 = 1.00
    Path("outputs/mgo_v103.out"),
    Path("outputs/mgo_v106.out"),
    Path("outputs/mgo_v109.out"),
]

file_list = write_file_list(OUTPUTS, "crystal-files.txt")
print(file_list.read_text())



## 5. Generate a normalized E–V dataset with Quantas

We ask Quantas to express the data using the **crystallographic conventional
cell**. For MgO, this makes the reported cubic cell parameter directly
comparable with the structure used in the course.


In [ ]:

!quantas eos inpgen crystal-files.txt \
    --interface crystal \
    --list \
    --crystal-reference crystallographic \
    --jobname "MgO static energy-volume EOS" \
    -o mgo_ev.dat


In [ ]:

dataset = read_quantas_ev_dataset("mgo_ev.dat")
dataset



### Question 4

Where is the lowest *sampled* energy? Is it associated with the optimized
reference volume, as you expected?

**Your answer:**  
_Write your answer here._



## 6. Fit a third-order Birch–Murnaghan EOS

The fit uses the public Quantas EOS API.


In [ ]:

fit = fit_bm3_with_quantas("mgo_ev.dat")

for name in ("E0", "V0", "K0", "KP"):
    print(f"{name:3s} = {fit.parameter_values[name]:.10g}")

print(f"RMSE = {fit.fit.rmse:.6e} Ha")



The parameters are:

- `E0`: fitted equilibrium energy;
- `V0`: fitted equilibrium volume;
- `K0`: equilibrium bulk modulus in GPa;
- `KP`: $K'_0$, the first pressure derivative of the bulk modulus.


In [ ]:

table = eos_teaching_table(
    dataset,
    fit,
    formula_units_per_cell=4,   # conventional MgO cell: 4 formula units
)
table



## 7. Plot the energy–volume relation


In [ ]:

plot_energy_volume(dataset, fit);



### Question 5

Compare the minimum of the fitted curve with the equilibrium volume obtained
from the direct geometry optimization.

Should the two estimates be similar? Why are they not mathematically required
to be numerically identical?

**Your answer:**  
_Write 3–4 lines here._



## 8. Reconstruct pressure from the energy EOS

The pressure is obtained from

$$
P(V)=-\frac{dE}{dV}.
$$


In [ ]:

plot_pressure_volume(dataset, fit);



### Question 6

At approximately which volume does the fitted pressure cross zero?

How is this related to the minimum of $E(V)$?

**Your answer:**  
_Write 2–3 lines here._



## 9. Interpret the bulk modulus

### Question 7

What does your fitted $K_0$ tell you physically about MgO?

Would a material with a larger $K_0$ be easier or harder to compress?

**Your answer:**  
_Write 2–3 lines here._



### Question 8

Why should we be cautious when comparing this static $K_0$ directly with an
experimental value measured at room temperature?

**Your answer:**  
_Write 2–3 lines here._



## 10. The modelling chain

Complete the interpretation in words:

```text
experimental structure
        ↓
static DFT optimization
        ↓
equilibrium structure
        ↓
fixed-volume calculations
        ↓
E(V)
        ↓
BM3
        ↓
P(V), V0, K0, K'0
```

### Question 9

In no more than five lines, explain how a set of microscopic
electronic-structure calculations has been turned into a macroscopic mechanical
property.

**Your answer:**  
_Write your answer here._



## Before you finish

Save the completed notebooks and all compact input/output files that belong to
the activity.

Then:

```text
save → commit → Sync Changes → verify on GitHub → stop Codespace
```
